# Local RAG on Google Colab
Free stack: Ollama (Llama 3.2) + sentence-transformers + ChromaDB.

**Tip:** Runtime > Change runtime type > T4 GPU makes answers much faster.

## 1. Install Ollama and Python packages

In [1]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q chromadb sentence-transformers ollama pypdf

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

## 2. Start the Ollama server and download the model

In [2]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull llama3.2

## 3. Upload your documents (PDF, TXT, MD)

In [3]:
import os
from google.colab import files

os.makedirs("docs", exist_ok=True)
for name, data in files.upload().items():
    with open(f"docs/{name}", "wb") as f:
        f.write(data)
print(os.listdir("docs"))

Saving emma_watson.pdf to emma_watson.pdf
['emma_watson.pdf']


## 4. RAG code

In [4]:
from pathlib import Path
import chromadb, ollama
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "llama3.2"
CHUNK_SIZE, OVERLAP, TOP_K = 800, 150, 4
SUPPORTED = {".pdf", ".txt", ".md"}

embedder = SentenceTransformer(EMBED_MODEL)
db = chromadb.PersistentClient(path="chroma_db")
col = db.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

def read_file(path):
    if path.suffix.lower() == ".pdf":
        return "\n".join(p.extract_text() or "" for p in PdfReader(str(path)).pages)
    return path.read_text(encoding="utf-8", errors="ignore")

def chunk(text):
    step = CHUNK_SIZE - OVERLAP
    chunks = [text[i:i + CHUNK_SIZE].strip() for i in range(0, len(text), step)]
    return [c for c in chunks if c]

def ingest(folder):
    for path in Path(folder).rglob("*"):
        if path.suffix.lower() not in SUPPORTED:
            continue
        chunks = chunk(read_file(path))
        if not chunks:
            print(f"Skipped {path} (no extractable text)")
            continue
        col.upsert(
            ids=[f"{path}:{i}" for i in range(len(chunks))],
            documents=chunks,
            embeddings=embedder.encode(chunks).tolist(),
            metadatas=[{"source": str(path)}] * len(chunks),
        )
        print(f"Indexed {path} ({len(chunks)} chunks)")

def ask(question):
    q_emb = embedder.encode([question]).tolist()
    hits = col.query(query_embeddings=q_emb, n_results=TOP_K)
    context = "\n\n---\n\n".join(hits["documents"][0])
    prompt = (
        "Answer using ONLY the context below. "
        "If the answer isn't there, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    reply = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}])
    print(reply["message"]["content"])
    print("\nSources:", {m["source"] for m in hits["metadatas"][0]})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5. Index and ask

In [5]:
ingest("docs")

Indexed docs/emma_watson.pdf (6 chunks)


In [12]:
ask("Answer in exactly 5 words: where was Emma born?")

Paris, France on April 15th.

Sources: {'docs/emma_watson.pdf'}


In [ ]:
import os
from getpass import getpass

# Securely prompt for your GitHub Personal Access Token at runtime
github_token = getpass("Enter your GitHub Personal Access Token: ")

repo = "VigneshA7MSD/Emma_Rag_Ollama"
url = f"https://{github_token}@github.com/{repo}.git"

!printf "chroma_db/\nollama.log\nsample_data/\n" > .gitignore
!git init -q -b main && git config user.email "me@example.com" && git config user.name "me"
!git add -A && git commit -qm "Add RAG project"
!git remote add origin {url} 2>/dev/null; git remote set-url origin {url}; git push -u origin main